In [ ]:
# ============================================================
# Cell 0: RFdiffusion environment setup (run once per session)
# ============================================================

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

!git clone https://github.com/RosettaCommons/RFdiffusion.git
%cd /content/RFdiffusion

# Core dependencies
!pip install -q hydra-core omegaconf pyrsistent biopython e3nn
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html

# RFdiffusion
!pip install -e .

# SE3 transformer dependency
!git clone https://github.com/NVIDIA/DeepLearningExamples.git
%cd /content/RFdiffusion/DeepLearningExamples/DGLPyTorch/DrugDiscovery/SE3Transformer
!pip install -e .

%cd /content/RFdiffusion

# model weights
!mkdir -p models
!wget -nc -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt
!wget -nc -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt

print("✓ RFdiffusion setup complete")


GPU available: True
GPU: Tesla T4
Cloning into 'RFdiffusion'...
remote: Enumerating objects: 753, done.
remote: Counting objects: 100% (546/546), done.
remote: Compressing objects: 100% (340/340), done.
remote: Total 753 (delta 337), reused 238 (delta 202), pack-reused 207 (from 1)
Receiving objects: 100% (753/753), 10.19 MiB | 19.18 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/RFdiffusion
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 37.5 MB/s eta 0:00:00
Looking in links: https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 MB 3.8 MB/s eta 0:00:00
Obtaining file:///content/RFdiffusion
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple version

In [ ]:
# ============================================================
# Cell 1: Connect project files
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR="/content/drive/Othercomputers/My Mac/pdl1-mini-binder-design"
TARGET_PDB=f"{PROJECT_DIR}/data/structures/pdl1_clean.pdb"

!ls "$TARGET_PDB"

print("✓ Project connected")


Mounted at /content/drive
'/content/drive/Othercomputers/My Mac/pdl1-mini-binder-design/data/structures/pdl1_clean.pdb'
✓ Project connected


In [ ]:
# ============================================================
# Cell 2: Inspect target residue numbering
# ============================================================

from Bio.PDB import PDBParser

parser=PDBParser(QUIET=True)
structure=parser.get_structure("pdl1",TARGET_PDB)

for chain in structure[0]:
    residues=[r for r in chain.get_residues() if r.id[0]==" "]
    print(f"Chain {chain.id}: {residues[0].id[1]} -> {residues[-1].id[1]} ({len(residues)} residues)")


Chain A: 18 -> 132 (115 residues)


In [ ]:
# ============================================================
# Cell 3: Generate binders
# ============================================================

%cd /content/RFdiffusion

import os, sys, subprocess
import glob

sys.path.append("/content/RFdiffusion")
os.environ["PYTHONPATH"]="/content/RFdiffusion:"+os.environ.get("PYTHONPATH","")

HOTSPOT_RESIDUES="A18,A19,A20,A23,A26,A56,A66,A76,A113,A115,A120,A122,A123,A124,A125"
TARGET_RANGE="A18-132"   # update based on Cell 2
BINDER_LENGTH=70
NUM_DESIGNS=24

# FIXED: Write directly to Google Drive so designs survive disconnects
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR="/content/drive/MyDrive/Colab Notebooks/rfdiffusion_outputs"
!mkdir -p "$OUTPUT_DIR"

# Check what's already done ONCE, before the loop
existing = set(
    os.path.basename(f)
    for f in glob.glob(f"{OUTPUT_DIR}/design_*.pdb")
    if "traj" not in f
)
print(f"Resuming — {len(existing)} designs already complete")

for i in range(NUM_DESIGNS):
    # FIXED: Actually skip completed designs
    expected_file = f"design_{i}_0.pdb"
    if expected_file in existing:
        continue

    cmd=[
        "python",
        "scripts/run_inference.py",
        f"inference.output_prefix={OUTPUT_DIR}/design_{i}",
        f"inference.input_pdb={TARGET_PDB}",
        f"contigmap.contigs=[{BINDER_LENGTH}-{BINDER_LENGTH}/0 {TARGET_RANGE}]",
        f"ppi.hotspot_res=[{HOTSPOT_RESIDUES}]",
        "inference.num_designs=1",
        "denoiser.noise_scale_ca=0",
        "denoiser.noise_scale_frame=0",
    ]

    result=subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=os.environ.copy()
    )

    if result.returncode!=0:
        print(f"Design {i} failed")
        print(result.stderr[-1000:])
        break

    if i%10==0:
        print(f"Completed {i+1}/{NUM_DESIGNS}")

print("✓ Binder generation complete")

/content/RFdiffusion
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Resuming — 0 designs already complete
Completed 1/24
Completed 11/24
Completed 21/24
✓ Binder generation complete


In [ ]:
# ============================================================
# Cell 4: Quick quality filter on backbone geometry
# ============================================================
!pip install -q biopython
import os
import glob
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser, NeighborSearch
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR="/content/drive/MyDrive/Colab Notebooks/rfdiffusion_outputs"
parser = PDBParser(QUIET=True)

design_files = sorted(glob.glob(f"{OUTPUT_DIR}/design_*.pdb"))
design_files = [f for f in design_files if "traj" not in f]

print(f"Found {len(design_files)} design PDB files")

# Inspect first design
if design_files:
    test_structure = parser.get_structure("test", design_files[0])
    print("\nChain inventory for first design:")
    for chain in test_structure[0]:
        residues = [r for r in chain.get_residues() if r.id[0] == " "]
        if residues:
            print(
                f"Chain {chain.id}: {len(residues)} residues "
                f"({residues[0].id[1]}–{residues[-1].id[1]})"
            )

quality_data = []

for pdb_file in design_files:

    try:
        structure = parser.get_structure("design", pdb_file)
        model = structure[0]

        # Identify binder vs target by residue count
        # Binder = 70 residues (what you designed)
        # Target = 115 residues (PD-L1)
        chains = list(model.get_chains())
        chain_sizes = []
        for c in chains:
            res = [r for r in c.get_residues() if r.id[0] == ' ']
            chain_sizes.append((c, len(res)))

        # Binder is the shorter chain (your designed 70-mer)
        chain_sizes.sort(key=lambda x: x[1])
        binder_chain = chain_sizes[0][0]
        target_chain = chain_sizes[-1][0]

        binder_residues = [
            r for r in binder_chain.get_residues()
            if r.id[0] == " "
        ]

        target_residues = [
            r for r in target_chain.get_residues()
            if r.id[0] == " "
        ]

        n_res = len(binder_residues)

        ca_coords = np.array([
            r["CA"].get_vector().get_array()
            for r in binder_residues
            if "CA" in r
        ])

        if len(ca_coords) == 0:
            raise ValueError("No CA atoms found in binder chain")

        centroid = ca_coords.mean(axis=0)
        rg = np.sqrt(np.mean(np.sum((ca_coords - centroid) ** 2, axis=1)))

        target_atoms = list(target_chain.get_atoms())
        ns = NeighborSearch(target_atoms)

        contacting_binder_residues = set()
        contacting_target_residues = set()
        atom_contacts = 0

        for residue in binder_residues:
            for atom in residue.get_atoms():
                neighbors = ns.search(atom.get_vector().get_array(), 4.5)

                if neighbors:
                    atom_contacts += len(neighbors)
                    contacting_binder_residues.add(residue.id[1])

                    for n_atom in neighbors:
                        parent_res = n_atom.get_parent()
                        if parent_res.id[0] == " ":
                            contacting_target_residues.add(parent_res.id[1])

        quality_data.append({
            "file": os.path.basename(pdb_file),
            "binder_chain": binder_chain.id,
            "target_chain": target_chain.id,
            "n_residues": n_res,
            "radius_of_gyration": round(rg, 2),
            "atom_contacts": atom_contacts,
            "binder_contact_residues": len(contacting_binder_residues),
            "target_contact_residues": len(contacting_target_residues),
        })

    except Exception as e:
        print(f"Error processing {pdb_file}: {e}")

df_designs = pd.DataFrame(quality_data)

df_good = df_designs[
    (df_designs["radius_of_gyration"] > 8) &
    (df_designs["radius_of_gyration"] < 20) &
    (df_designs["binder_contact_residues"] >= 3) &
    (df_designs["target_contact_residues"] >= 2)
]

print(f"\nDesigns passing geometry filter: {len(df_good)}/{len(df_designs)}")

print("\nTop designs by target contact residues:")
print(
    df_good
    .sort_values(
        ["target_contact_residues", "binder_contact_residues", "atom_contacts"],
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

df_designs.to_csv(f"{OUTPUT_DIR}/all_design_geometry_metrics.csv", index=False)
df_good.to_csv(f"{OUTPUT_DIR}/filtered_designs.csv", index=False)

print(df_designs[['file', 'radius_of_gyration', 'binder_contact_residues', 'target_contact_residues']].to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 24 design PDB files

Chain inventory for first design:
Chain B: 70 residues (1–70)
Chain A: 115 residues (18–132)

Designs passing geometry filter: 4/24

Top designs by target contact residues:
           file binder_chain target_chain  n_residues  radius_of_gyration  atom_contacts  binder_contact_residues  target_contact_residues
design_14_0.pdb            B            A          70               15.86             16                        5                        4
 design_1_0.pdb            B            A          70               11.95              4                        3                        3
design_20_0.pdb            B            A          70               13.46              7                        3                        2
 design_2_0.pdb            B            A          70               11.90              3                        3  

In [ ]:
# ============================================================
# Cell 5: Download results
# ============================================================
# Download filtered designs to your local machine for analysis/archiving

import shutil

# Zip the filtered designs
filtered_files = df_good['file'].tolist()
!mkdir -p outputs/filtered_backbones
for f in filtered_files:
    shutil.copy(f"{OUTPUT_DIR}/{f}", "outputs/filtered_backbones/")

shutil.make_archive("filtered_backbones", 'zip', "outputs/filtered_backbones")

from google.colab import files
files.download("filtered_backbones.zip")
files.download(f"{OUTPUT_DIR}/filtered_designs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>